In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re
import shutil


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [4]:
result_from_prev_analysis = pd.read_parquet("/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/results/results_myvideos_174_rec_session_2/logits_output_meta.parquet")
print(result_from_prev_analysis.shape)
del result_from_prev_analysis['top5_classes']
result_from_prev_analysis.head(4)

(216, 12)


,video_id,cam,session,template,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,260930-122254,cam0,plug_2,Putting [something] next to [something],107,107,Putting [something] next to [something],0.41,"[107, 114, 42, 19, 109]",True,True
1,260930-122254,cam1,plug_2,Putting [something] next to [something],107,107,Putting [something] next to [something],0.74,"[107, 19, 114, 42, 109]",True,True
2,260930-122254,cam2,plug_2,Putting [something] next to [something],107,105,Putting [something] in front of [something],0.75,"[105, 18, 114, 42, 125]",False,False
3,260930-122317,cam0,plug_2,Putting [something] next to [something],107,107,Putting [something] next to [something],0.41,"[107, 42, 109, 41, 112]",True,True


In [6]:
result = pd.read_parquet("./predictions.parquet")
print(result.shape)
result.sample(2)

(216, 4)


,video_id,label,top1_label,correct_top1
36,260930-122916_cam0,101,101,True
1,260930-122254_cam1,109,73,False


In [7]:
# rename top1_label to top1_pred
result = result.rename(columns={'top1_label':'top1_pred'})

# I want to break video_id into video_id and cam
result[['video_id', 'cam']] = result.video_id.str.split('_', expand=True)
print(result.shape)

# Bring in session column with a left join
# Bring in template column with a left join (to later verify match rate with my custom template2id mapping.)
result_from_prev_analysis = result_from_prev_analysis[['video_id','cam', 'session','template']]
result = pd.merge(result, result_from_prev_analysis, how = 'left', on = ['video_id','cam']) # LEFT JOIN
print(result.shape)

result.sample(3)

(216, 5)
(216, 7)


,video_id,label,top1_pred,correct_top1,cam,session,template
79,260930-125220,114,114,True,cam1,lady_2,Putting [something] onto [something]
49,260930-124846,109,109,True,cam1,lady_2,Putting [something] next to [something]
198,260930-145307,1,157,False,cam0,attaching_1,Attaching [something] to [something]


In [8]:
d = np.load("./predictions.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(216, 174)


NpzFile './predictions.npz' with keys: logits, probs, video_ids, labels

# main result

In [9]:

n = len(result)
top1_acc  = result["correct_top1"].mean()

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 4),
}
my_metrics

{'total_evaluated': 216, 'top1_accuracy': np.float64(0.287)}

- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [10]:
result.groupby("cam")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
cam,
cam0,0.36
cam1,0.38
cam2,0.12


In [11]:
result.groupby("template")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
).sort_values("template")

,top1_acc
template,
Attaching [something] to [something],0.11
Moving [something] up,0.22
Piling [something] up,0.11
Plugging [something] into [something],0.56
Pushing [something] with [something],0.44
Putting [something] next to [something],0.39
Putting [something] on a surface,0.06
Putting [something] onto [something],0.36
Showing a photo of [something] to the camera,0.22


In [12]:
result[result.cam=='cam1'].groupby("template")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
).sort_values("template")

,top1_acc
template,
Attaching [something] to [something],0.00
Moving [something] up,0.17
Piling [something] up,0.00
Plugging [something] into [something],0.67
Pushing [something] with [something],0.58
Putting [something] next to [something],0.67
Putting [something] on a surface,0.08
Putting [something] onto [something],0.42
Showing a photo of [something] to the camera,0.67


In [23]:
result[result.cam=='cam1'].groupby("session")[["correct_top1"]].mean().rename(
        columns={"correct_top1": "top1_acc"}
)

,top1_acc
session,
attaching_1,0.000000
books_2,0.333333
lady_2,0.333333
magnet_2,0.466667
piling_1,0.000000
plug_2,0.400000
plugging_1,0.666667
showing_photo_of_1,0.666667


# Granular analysis

##### Distractor objects -> expected spike in action/verb -- P(Action|Noun) estimate
- Plug -> Plugging [something] into [something] -- 55%
- lady/person -> Showing a photo of [something] to the camera -- 96%
- magnet -> Attaching [something] to [something] -- 60%
- Books -> Piling [something] up -- 55%
- (~) Paper -> Tearing [something] into two pieces -- 25%

##### id2label

In [13]:
# Load mapping text file into a Series with template strings as the index
template2id_mapping = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/experiment_ssv2_object_occlusion/data/template2id_mapping.txt', sep='\t', header=None, index_col=1)[0]
template2id_mapping

1
Approaching [something] with your camera                                 0
Attaching [something] to [something]                                     1
Bending [something] so that it deforms                                   2
Bending [something] until it breaks                                      3
Burying [something] in [something]                                       4
                                                                      ... 
[Something] being deflected from [something]                           169
[Something] colliding with [something] and both are being deflected    170
[Something] colliding with [something] and both come to a halt         171
[Something] falling like a feather or paper                            172
[Something] falling like a rock                                        173
Name: 0, Length: 174, dtype: int64

In [14]:
# Open text file into a dict with label string as key and integer ID as value
with open('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/experiment_ssv2_object_occlusion/data/template2id_mapping.txt') as f:
    id2label_dict = {int(line.strip().split('\t')[0]) : line.strip().split('\t')[1] for line in f}

id2label_dict

{0: 'Approaching [something] with your camera',
 1: 'Attaching [something] to [something]',
 2: 'Bending [something] so that it deforms',
 3: 'Bending [something] until it breaks',
 4: 'Burying [something] in [something]',
 5: 'Closing [something]',
 6: 'Covering [something] with [something]',
 7: 'Digging [something] out of [something]',
 8: 'Dropping [something] behind [something]',
 9: 'Dropping [something] in front of [something]',
 10: 'Dropping [something] into [something]',
 11: 'Dropping [something] next to [something]',
 12: 'Dropping [something] onto [something]',
 13: 'Failing to put [something] into [something] because [something] does not fit',
 14: 'Folding [something]',
 15: 'Hitting [something] with [something]',
 16: 'Holding [something]',
 17: 'Holding [something] behind [something]',
 18: 'Holding [something] in front of [something]',
 19: 'Holding [something] next to [something]',
 20: 'Holding [something] over [something]',
 21: 'Laying [something] on the table on 

In [15]:
# Apply mapping to create new column
result['template_custom'] = result['label'].map(id2label_dict)
result.sample(7)

,video_id,label,top1_pred,correct_top1,cam,session,template,template_custom
19,260930-122448,44,18,False,cam1,plug_2,Moving [something] up,Moving [something] up
151,260930-143505,111,114,False,cam1,books_2,Putting [something] on a surface,Putting [something] on a surface
59,260930-124955,111,2,False,cam2,lady_2,Putting [something] on a surface,Putting [something] on a surface
144,260930-143429,111,139,False,cam0,books_2,Putting [something] on a surface,Putting [something] on a surface
62,260930-125006,111,31,False,cam2,lady_2,Putting [something] on a surface,Putting [something] on a surface
177,260930-143810,101,101,True,cam0,books_2,Pushing [something] with [something],Pushing [something] with [something]
118,260930-142029,114,75,False,cam1,magnet_2,Putting [something] onto [something],Putting [something] onto [something]


In [16]:
pd.testing.assert_series_equal(result.template, result.template_custom, check_names=False, check_exact=False)

In [17]:
np.mean(result.template == result.template_custom)*100

np.float64(100.0)

In [18]:
label2id_dict = {v: k for k, v in id2label_dict.items()}

print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[(0, 'Approaching [something] with your camera'), (1, 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', 0), ('Attaching [something] to [something]', 1)]


In [19]:
result.head(1)

,video_id,label,top1_pred,correct_top1,cam,session,template,template_custom
0,260930-122254,109,18,False,cam0,plug_2,Putting [something] next to [something],Putting [something] next to [something]


In [47]:
df = result[['video_id',	'cam'	,'session'	,'template',	'label'	,'top1_pred','correct_top1']]
print(df.shape)
df.head(2)

(216, 7)


,video_id,cam,session,template,label,top1_pred,correct_top1
0,260930-122254,cam0,plug_2,Putting [something] next to [something],109,18,False
1,260930-122254,cam1,plug_2,Putting [something] next to [something],109,73,False


In [48]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,template,label,top1_pred,correct_top1,prob_0,prob_1,prob_2,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,260930-122254,cam0,plug_2,Putting [something] next to [something],109,18,False,0.000088,0.014976,0.000190,...,0.000009,0.001523,0.000631,0.000092,0.000526,0.000405,0.000684,0.001208,0.000002,0.000058
1,260930-122254,cam1,plug_2,Putting [something] next to [something],109,73,False,0.000116,0.008115,0.000276,...,0.000017,0.001032,0.001395,0.000062,0.000153,0.000305,0.000236,0.000354,0.000005,0.000197


#### Main Analysis 
Comparison of:
- P(label=3|session=plug_1) -- should be high.
- P(label=3|session=plug_2) -- if its high it means plug as distractor is increasing probabilty of label=3.
- P(label=3|session not like 'plug%')

In [49]:
pd.set_option('display.float_format',  '{:,.5f}'.format)

In [50]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        # top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/apply_dinov2_attn_probe/data/myvideos_2/frames/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/vjepa_evaluation_on_ssv2/analysis_videos_visualized/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = (dist_df['video_id'] + '_' + dist_df['cam']).dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

# Analysis special

### "special objects in common actions":

- plug_2 shows 15 videos of common actions being performed with object plug. We should see a spike in 'Plugging [something] into [something]' for these. 
- So plug_2 is actually where plug is a distractor.

- I have plug_2, lady_2, magnet_2, books_2 for "special objects in common actions" in recording session 2

In [51]:
df.session.unique()

array(['plug_2', 'lady_2', 'magnet_2', 'books_2', 'plugging_1',
       'showing_photo_of_1', 'attaching_1', 'piling_1'], dtype=object)

In [52]:
avg_prob_in_diff_cases(df, session_type='plug_2', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plug_2 =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc                                     examples
0  plug_2      15    0.08055   0.12272  0.40000  260930-122254, 260930-122317, 260930-122340
==================== Videos ====================
        video_id   cam session                                 template label top1_pred correct_top1 prob_73 prob_19 prob_18 prob_109 prob_101
1  260930-122254  cam1  plug_2  Putting [something] next to [something]   109        73        False 0.14910 0.14373 0.11910  0.10089  0.05036

Top 5 Probability Breakdown:
  prob_73  ->  0.14910  |  Pretending to put [something] next to [something]
  prob_19  ->  0.14373  |  Holding [something] next to [something]
  prob_18  ->  0.11910  |  Holding [something] in front of [something]
  prob_109 ->  0.10089  |  Putting [something] next to [something]
  prob_101 ->  0.05036  |  Pushing [some

In [ ]:
avg_prob_in_diff_cases(df, session_type='lady_2', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= lady_2 =========================
label 128: Showing a photo of [something] to the camera
cams used: ['cam1']
  session  videos  prob_mean  prob_std  top1acc                                     examples
0  lady_2      15    0.00458   0.01003  0.33333  260930-124820, 260930-124846, 260930-124905
==================== Videos ====================
         video_id   cam session                                 template label top1_pred correct_top1 prob_109 prob_73 prob_39 prob_141 prob_36
46  260930-124820  cam1  lady_2  Putting [something] next to [something]   109       109         True  0.44200 0.24408 0.04569  0.03827 0.03522

Top 5 Probability Breakdown:
  prob_109 ->  0.44200  |  Putting [something] next to [something]
  prob_73  ->  0.24408  |  Pretending to put [something] next to [something]
  prob_39  ->  0.04569  |  Moving [something] away from [something]
  prob_141 ->  0.03827  |  Taking [one of many similar things on the table]
  prob_36  ->  0.03522  

In [55]:
avg_prob_in_diff_cases(df, session_type='magnet_2', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= magnet_2 =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
    session  videos  prob_mean  prob_std  top1acc                                     examples
0  magnet_2      15    0.00873   0.02341  0.46667  260930-141745, 260930-141807, 260930-141829
==================== Videos ====================
         video_id   cam   session                                 template label top1_pred correct_top1 prob_109 prob_73 prob_39 prob_36 prob_120
91  260930-141745  cam1  magnet_2  Putting [something] next to [something]   109       109         True  0.18433 0.16859 0.11220 0.10257  0.08407

Top 5 Probability Breakdown:
  prob_109 ->  0.18433  |  Putting [something] next to [something]
  prob_73  ->  0.16859  |  Pretending to put [something] next to [something]
  prob_39  ->  0.11220  |  Moving [something] away from [something]
  prob_36  ->  0.10257  |  Moving [something] and [something] closer to each other
  prob_120 ->  0.

In [56]:
avg_prob_in_diff_cases(df, session_type='books_2', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= books_2 =========================
label 48: Piling [something] up
cams used: ['cam1']
   session  videos  prob_mean  prob_std  top1acc                                     examples
0  books_2      15    0.01343   0.02126  0.33333  260930-143319, 260930-143358, 260930-143410
==================== Videos ====================
          video_id   cam  session                                 template label top1_pred correct_top1 prob_125 prob_109 prob_124 prob_73 prob_11
136  260930-143319  cam1  books_2  Putting [something] next to [something]   109       125        False  0.46084  0.16823  0.14164 0.05676 0.04948

Top 5 Probability Breakdown:
  prob_125 ->  0.46084  |  Showing [something] next to [something]
  prob_109 ->  0.16823  |  Putting [something] next to [something]
  prob_124 ->  0.14164  |  Showing [something] behind [something]
  prob_73  ->  0.05676  |  Pretending to put [something] next to [something]
  prob_11  ->  0.04948  |  Dropping [something] ne

### "common objects in special actions: ":

- plugging_1 shows 3 videos of special action "Plugging [something] into [something] " with common objects (not a plug)
- To verify if everything is working as intended for these special actions.

- I have plugging_1, showing_1, attaching_1, piling_1 for "common objects in special actions" in recording session 2

In [57]:
avg_prob_in_diff_cases(df, session_type='plugging', label='Plugging [something] into [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= plugging =========================
label 49: Plugging [something] into [something]
cams used: ['cam1']
      session  videos  prob_mean  prob_std  top1acc                                     examples
0  plugging_1       3    0.40184   0.27166  0.66667  260930-144612, 260930-144637, 260930-144729
==================== Videos ====================
          video_id   cam     session                               template label top1_pred correct_top1 prob_49 prob_50  prob_1 prob_18 prob_165
181  260930-144612  cam1  plugging_1  Plugging [something] into [something]    49        49         True 0.57648 0.18337 0.08399 0.06508  0.02071

Top 5 Probability Breakdown:
  prob_49  ->  0.57648  |  Plugging [something] into [something]
  prob_50  ->  0.18337  |  Plugging [something] into [something] but pulling it right out as you remove your hand
  prob_1   ->  0.08399  |  Attaching [something] to [something]
  prob_18  ->  0.06508  |  Holding [something] in front of [som

In [58]:
avg_prob_in_diff_cases(df, session_type='showing', label='Showing a photo of [something] to the camera', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= showing =========================
label 128: Showing a photo of [something] to the camera
cams used: ['cam1']
              session  videos  prob_mean  prob_std  top1acc                                     examples
0  showing_photo_of_1       3    0.69809   0.48509  0.66667  260930-145029, 260930-145050, 260930-145114
==================== Videos ====================
          video_id   cam             session                                      template label top1_pred correct_top1 prob_128 prob_44  prob_6 prob_124 prob_144
190  260930-145029  cam1  showing_photo_of_1  Showing a photo of [something] to the camera   128       128         True  0.97048 0.00216 0.00195  0.00158  0.00156

Top 5 Probability Breakdown:
  prob_128 ->  0.97048  |  Showing a photo of [something] to the camera
  prob_44  ->  0.00216  |  Moving [something] up
  prob_6   ->  0.00195  |  Covering [something] with [something]
  prob_124 ->  0.00158  |  Showing [something] behind [somethin

In [59]:
avg_prob_in_diff_cases(df, session_type='attaching', label='Attaching [something] to [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= attaching =========================
label 1: Attaching [something] to [something]
cams used: ['cam1']
       session  videos  prob_mean  prob_std  top1acc                                     examples
0  attaching_1       3    0.02162   0.01080  0.00000  260930-145307, 260930-145329, 260930-145941
==================== Videos ====================
          video_id   cam      session                              template label top1_pred correct_top1 prob_157 prob_90 prob_64 prob_165  prob_3
199  260930-145307  cam1  attaching_1  Attaching [something] to [something]     1       157        False  0.51587 0.10068 0.05204  0.04569 0.03530

Top 5 Probability Breakdown:
  prob_157 ->  0.51587  |  Trying to bend [something unbendable] so nothing happens
  prob_90  ->  0.10068  |  Pulling two ends of [something] but nothing happens
  prob_64  ->  0.05204  |  Pretending or trying and failing to twist [something]
  prob_165 ->  0.04569  |  Twisting [something]
  prob_3   

In [60]:
avg_prob_in_diff_cases(df, session_type='piling', label='Piling [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= piling =========================
label 48: Piling [something] up
cams used: ['cam1']
    session  videos  prob_mean  prob_std  top1acc                                     examples
0  piling_1       3    0.03931   0.01101  0.00000  260930-150113, 260930-150230, 260930-150433
==================== Videos ====================
          video_id   cam   session               template label top1_pred correct_top1 prob_114 prob_139 prob_75 prob_121 prob_111
208  260930-150113  cam1  piling_1  Piling [something] up    48       114        False  0.31333  0.20674 0.06180  0.05066  0.03119

Top 5 Probability Breakdown:
  prob_114 ->  0.31333  |  Putting [something] onto [something]
  prob_139 ->  0.20674  |  Stacking [number of] [something]
  prob_75  ->  0.06180  |  Pretending to put [something] onto [something]
  prob_121 ->  0.05066  |  Removing [something], revealing [something] behind
  prob_111 ->  0.03119  |  Putting [something] on a surface

  prob_48  ->  0.0289